# 06 — Sprint 5: Model Diagnostics

This notebook explores the Sprint 5 model-diagnostic outputs produced by the
rolling-origin backtesting pipeline. It loads persisted CSV files from
`outputs/diagnostics/`, interprets each diagnostic component, and assembles
a report-level model-health summary.

**What this notebook does NOT do:**
- Rerun any part of the pipeline.
- Generate synthetic fallback data.
- Call any external API or LLM.
- Alter model-selection rankings (MASE remains the primary selection metric).
- Trigger automatic retraining.

## Business Objective

The forecasting pipeline selects one model per report based on out-of-sample MASE
computed across rolling backtest folds. Sprint 5 adds a **diagnostic layer** that
answers the operational question: *given the model that was selected, how well does
it behave?*

Diagnostics answer five specific questions:

| # | Component | Operational question |
|---|-----------|---------------------|
| 1 | Autocorrelation | Is temporal structure left unexplained in residuals? |
| 2 | Bias stability | Is the model systematically over- or under-forecasting? |
| 3 | Outlier detection | How frequent and severe are large misses? |
| 4 | Distribution shape | Are residuals strongly skewed or heavy-tailed? |
| 5 | Interval calibration | Are prediction intervals achieving nominal coverage? |

The five components are consolidated into a **model-health status** per report
(`healthy`, `watch`, `poor`, `insufficient_evidence`, `calculation_failed`).

**Key constraint:** diagnostics do NOT alter model selection. The selected model
remains the same. A `poor` health rating signals that a human reviewer should
investigate, not that automatic retraining has occurred.

## Diagnostic Architecture

```
Pipeline outputs
  outputs/metrics/backtest_predictions_latest.csv       (primary residual source)
  outputs/metrics/realized_forecast_history.csv         (production errors)
  outputs/metrics/backtest_metrics_latest.csv           (model selection metrics)
        |
        v
Sprint 5 diagnostic modules  (src/models/)
  residual_datasets.py           -- extracts residuals from three sources
  autocorrelation_diagnostics.py -- ACF, Ljung-Box, Durbin-Watson
  bias_stability_diagnostics.py  -- mean/median bias, variance stability
  outlier_distribution_diagnostics.py -- outlier rates, distribution shape
  interval_calibration_diagnostics.py -- coverage, width, Winkler score
  model_health.py                -- consolidates into one row per report
        |
        v
outputs/diagnostics/
  training_residuals_latest.csv
  backtest_forecast_errors_latest.csv
  production_forecast_errors_latest.csv
  training_autocorrelation_diagnostics_latest.csv
  backtest_autocorrelation_by_fold_latest.csv
  backtest_autocorrelation_summary_latest.csv
  production_autocorrelation_diagnostics_latest.csv
  training_bias_stability_diagnostics_latest.csv
  backtest_bias_stability_by_fold_latest.csv
  backtest_bias_stability_summary_latest.csv
  production_bias_stability_diagnostics_latest.csv
  training_outlier_distribution_diagnostics_latest.csv
  backtest_outlier_distribution_by_fold_latest.csv
  backtest_outlier_distribution_summary_latest.csv
  production_outlier_distribution_diagnostics_latest.csv
  backtest_interval_calibration_by_fold_latest.csv
  backtest_interval_calibration_summary_latest.csv
  production_interval_calibration_latest.csv
  report_model_diagnostics_latest.csv              <- consolidated model health
        |
        v
Downstream consumers
  Notebook 07 (user analytics), Notebook 08 (GenAI insights), Streamlit app
```

**Evidence-sufficiency rules:**
- `MIN_VALID_BACKTEST_FOLDS = 2`: fewer than 2 valid backtest folds → `insufficient_evidence`.
- `MIN_PRODUCTION_ERROR_COUNT = 10`: fewer than 10 realized production observations → production diagnostics not used as primary basis.

## Residual Sign Conventions

All three residual datasets use the **diagnostic residual convention**:

```
residual = actual - forecast_or_fitted
```

| Residual sign | Interpretation |
|--------------|----------------|
| positive | Model **underforecast** (actual > forecast) |
| negative | Model **overforecast** (actual < forecast) |
| zero | Perfect point forecast |

The canonical production monitoring history uses the **signed-error convention**:

```
signed_error = forecast - actual   (positive = overforecast)
```

For `production_forecast_errors_latest.csv`, both columns are present:
- `signed_error` is preserved as-is from the production history.
- `residual = actual - forecast = -signed_error` is derived for diagnostic consistency.

**Do not confuse the two.** Charts labeled `residual` use `actual - forecast`. Charts
labeled `signed_error` use `forecast - actual`.

## Load and Validate Diagnostic Outputs

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ── Schema constants from Sprint 5 source modules ────────────────────────────
from src.models.residual_datasets import (
    TRAINING_RESIDUALS_COLS,
    BACKTEST_FORECAST_ERRORS_COLS,
    PRODUCTION_FORECAST_ERRORS_COLS,
)
from src.models.autocorrelation_diagnostics import (
    TRAINING_ACF_COLS,
    BACKTEST_FOLD_ACF_COLS,
    BACKTEST_SUMMARY_ACF_COLS,
    PRODUCTION_ACF_COLS,
)
from src.models.bias_stability_diagnostics import (
    TRAINING_BIAS_COLS,
    BACKTEST_FOLD_BIAS_COLS,
    BACKTEST_SUMMARY_BIAS_COLS,
    PRODUCTION_BIAS_COLS,
)
from src.models.outlier_distribution_diagnostics import (
    TRAINING_OUTLIER_COLS,
    BACKTEST_FOLD_OUTLIER_COLS,
    BACKTEST_SUMMARY_OUTLIER_COLS,
    PRODUCTION_OUTLIER_COLS,
)
from src.models.interval_calibration_diagnostics import (
    BACKTEST_FOLD_INTERVAL_COLS,
    BACKTEST_SUMMARY_INTERVAL_COLS,
    PRODUCTION_INTERVAL_COLS,
)
from src.models.model_health import MODEL_HEALTH_COLS

print("Imports complete.")

In [ ]:
# ── Project-root detection (works when run from project root or notebooks/) ──
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
DIAG_DIR = ROOT / 'outputs' / 'diagnostics'

print(f"PROJECT_ROOT : {ROOT.resolve()}")
print(f"DIAG_DIR     : {DIAG_DIR.resolve()}")
print(f"DIAG_DIR exists: {DIAG_DIR.exists()}")

In [ ]:
def load_diag(filename, schema_cols=None):
    """Load a diagnostic CSV; return empty DataFrame if file not found."""
    path = DIAG_DIR / filename
    if not path.exists():
        print(f"  Not found: {filename}")
        return pd.DataFrame(columns=schema_cols or [])
    df = pd.read_csv(path)
    print(f"  OK {filename}: {len(df):,} rows")
    return df


print("Loading all 19 diagnostic output files...")

# Residual datasets
train_res       = load_diag('training_residuals_latest.csv',                   TRAINING_RESIDUALS_COLS)
bt_errors       = load_diag('backtest_forecast_errors_latest.csv',             BACKTEST_FORECAST_ERRORS_COLS)
prod_errors     = load_diag('production_forecast_errors_latest.csv',           PRODUCTION_FORECAST_ERRORS_COLS)

# Autocorrelation diagnostics
train_acf       = load_diag('training_autocorrelation_diagnostics_latest.csv', TRAINING_ACF_COLS)
bt_acf_fold     = load_diag('backtest_autocorrelation_by_fold_latest.csv',     BACKTEST_FOLD_ACF_COLS)
bt_acf_summary  = load_diag('backtest_autocorrelation_summary_latest.csv',     BACKTEST_SUMMARY_ACF_COLS)
prod_acf        = load_diag('production_autocorrelation_diagnostics_latest.csv', PRODUCTION_ACF_COLS)

# Bias / variance stability diagnostics
train_bias      = load_diag('training_bias_stability_diagnostics_latest.csv',  TRAINING_BIAS_COLS)
bt_bias_fold    = load_diag('backtest_bias_stability_by_fold_latest.csv',      BACKTEST_FOLD_BIAS_COLS)
bt_bias_summary = load_diag('backtest_bias_stability_summary_latest.csv',      BACKTEST_SUMMARY_BIAS_COLS)
prod_bias       = load_diag('production_bias_stability_diagnostics_latest.csv', PRODUCTION_BIAS_COLS)

# Outlier / distribution diagnostics
train_outlier       = load_diag('training_outlier_distribution_diagnostics_latest.csv', TRAINING_OUTLIER_COLS)
bt_outlier_fold     = load_diag('backtest_outlier_distribution_by_fold_latest.csv',      BACKTEST_FOLD_OUTLIER_COLS)
bt_outlier_summary  = load_diag('backtest_outlier_distribution_summary_latest.csv',      BACKTEST_SUMMARY_OUTLIER_COLS)
prod_outlier        = load_diag('production_outlier_distribution_diagnostics_latest.csv', PRODUCTION_OUTLIER_COLS)

# Interval calibration diagnostics
bt_interval_fold    = load_diag('backtest_interval_calibration_by_fold_latest.csv',   BACKTEST_FOLD_INTERVAL_COLS)
bt_interval_summary = load_diag('backtest_interval_calibration_summary_latest.csv',   BACKTEST_SUMMARY_INTERVAL_COLS)
prod_interval       = load_diag('production_interval_calibration_latest.csv',         PRODUCTION_INTERVAL_COLS)

# Consolidated model-health summary
health_df = load_diag('report_model_diagnostics_latest.csv', MODEL_HEALTH_COLS)

print("\nFile loading complete.")

In [ ]:
# Summary of loaded data
file_summary = [
    ('training_residuals', len(train_res)),
    ('backtest_forecast_errors', len(bt_errors)),
    ('production_forecast_errors', len(prod_errors)),
    ('training_acf', len(train_acf)),
    ('backtest_acf_by_fold', len(bt_acf_fold)),
    ('backtest_acf_summary', len(bt_acf_summary)),
    ('production_acf', len(prod_acf)),
    ('training_bias', len(train_bias)),
    ('backtest_bias_by_fold', len(bt_bias_fold)),
    ('backtest_bias_summary', len(bt_bias_summary)),
    ('production_bias', len(prod_bias)),
    ('training_outlier', len(train_outlier)),
    ('backtest_outlier_by_fold', len(bt_outlier_fold)),
    ('backtest_outlier_summary', len(bt_outlier_summary)),
    ('production_outlier', len(prod_outlier)),
    ('backtest_interval_by_fold', len(bt_interval_fold)),
    ('backtest_interval_summary', len(bt_interval_summary)),
    ('production_interval', len(prod_interval)),
    ('report_model_health', len(health_df)),
]

summary_df = pd.DataFrame(file_summary, columns=['dataset', 'row_count'])
summary_df['status'] = summary_df['row_count'].apply(lambda n: 'loaded' if n > 0 else 'empty / not found')
print(summary_df.to_string(index=False))

## Representative Report Selection

Case studies in later sections use a **deterministic** selection strategy:
sort all reports matching a given health status by `report_id` and take the first.
This ensures reproducible notebook output across re-runs on the same data.

In [ ]:
def select_report_by_status(df, status, n=1):
    """Deterministically select first report matching status, sorted by report_id."""
    if df.empty or 'model_diagnostic_status' not in df.columns:
        return None
    matches = df[df['model_diagnostic_status'] == status].sort_values('report_id')
    if matches.empty:
        return None
    return matches.iloc[0]


if not health_df.empty:
    status_counts = health_df['model_diagnostic_status'].value_counts()
    print("Model-health status distribution:")
    print(status_counts.to_string())

    # Pre-select representative reports for each status
    rep_healthy  = select_report_by_status(health_df, 'healthy')
    rep_watch    = select_report_by_status(health_df, 'watch')
    rep_poor     = select_report_by_status(health_df, 'poor')
    rep_insuf    = select_report_by_status(health_df, 'insufficient_evidence')

    for label, rep in [('healthy', rep_healthy), ('watch', rep_watch),
                       ('poor', rep_poor), ('insufficient_evidence', rep_insuf)]:
        rid = rep['report_id'] if rep is not None else 'none available'
        print(f"  representative {label}: {rid}")
else:
    print("Model-health file not yet available. Skipping representative selection.")
    rep_healthy = rep_watch = rep_poor = rep_insuf = None

## Training Residuals vs Forecast Errors

Sprint 5 distinguishes three residual sources with different evidential weight:

| Source | File | Evidential weight | Notes |
|--------|------|-------------------|-------|
| Training residuals | `training_residuals_latest.csv` | Weakest — in-sample | May be optimistic due to overfitting |
| Backtest forecast errors | `backtest_forecast_errors_latest.csv` | Primary | Out-of-sample rolling-origin |
| Production forecast errors | `production_forecast_errors_latest.csv` | Strongest (when sufficient) | Real operational evidence; may be sparse early |

All three use `residual = actual - forecast` (positive = underforecast).

In [ ]:
def _source_summary(df, name, residual_col='residual'):
    if df.empty:
        print(f"{name}: no data available")
        return
    col = residual_col if residual_col in df.columns else None
    print(f"\n{name}")
    print(f"  rows           : {len(df):,}")
    if 'report_id' in df.columns:
        print(f"  unique reports : {df['report_id'].nunique()}")
    if col and df[col].notna().any():
        vals = df[col].dropna()
        print(f"  residual mean  : {vals.mean():.4f}")
        print(f"  residual std   : {vals.std():.4f}")
        print(f"  residual range : [{vals.min():.2f}, {vals.max():.2f}]")

_source_summary(train_res,  'Training residuals',           'residual')
_source_summary(bt_errors,  'Backtest forecast errors',     'residual')
_source_summary(prod_errors,'Production forecast errors',   'residual')

## Residual Time-Series Plots

Plotting residuals over time reveals drift, volatility changes, and
seasonal patterns that summary statistics miss.

In [ ]:
def _plot_residuals(df, date_col, residual_col, report_col, title):
    """Plot residuals with a zero line for a single representative report."""
    if df.empty or residual_col not in df.columns or date_col not in df.columns:
        print(f"{title}: no data to plot.")
        return

    # Use first report alphabetically if no representative was selected
    if report_col in df.columns:
        rid = df[report_col].dropna().sort_values().iloc[0] if not df.empty else None
        sub = df[df[report_col] == rid].copy() if rid else df.copy()
        label = f"report_id={rid}"
    else:
        sub = df.copy()
        label = 'all'

    sub[date_col] = pd.to_datetime(sub[date_col], errors='coerce')
    sub = sub.dropna(subset=[date_col, residual_col]).sort_values(date_col)
    if sub.empty:
        print(f"{title}: no valid rows after date parsing.")
        return

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(sub[date_col], sub[residual_col], lw=0.8, color='steelblue', alpha=0.8)
    ax.axhline(0, color='black', lw=1, ls='--')
    ax.set_title(f"{title}\n{label}", fontsize=10)
    ax.set_xlabel(date_col)
    ax.set_ylabel('residual (actual − forecast)')
    plt.tight_layout()
    plt.show()


# Training residuals time-series
_plot_residuals(
    train_res,
    date_col='residual_date',
    residual_col='residual',
    report_col='report_id',
    title='Training residuals over time',
)

# Backtest forecast errors time-series
_plot_residuals(
    bt_errors,
    date_col='forecast_date',
    residual_col='residual',
    report_col='report_id',
    title='Backtest forecast errors over time',
)

# Production forecast errors time-series
_plot_residuals(
    prod_errors,
    date_col='forecast_date',
    residual_col='residual',
    report_col='report_id',
    title='Production forecast errors over time',
)

## Autocorrelation Diagnostics

Autocorrelation in residuals indicates that the model has not fully captured
temporal structure. Sprint 5 computes:

- **Lag-1 ACF**: short-lag correlation; AR structure left in residuals.
- **ACF at selected_m**: correlation at the seasonal lag `m` selected for that report.
- **Max ACF over tested lags**: worst-case autocorrelation.
- **Ljung-Box p-value**: joint significance test over multiple lags.
- **Durbin-Watson statistic**: first-order serial correlation (2 = none, <1 = positive).

**Interpretation note:** a significant Ljung-Box p-value (< 0.05) means residual
autocorrelation is detectable, but does NOT mean the model is unusable. For short
series or models with few parameters, some residual structure is expected. The
practical severity depends on the magnitude of ACF coefficients, not the p-value alone.

In [ ]:
def _show_acf_summary(df, label, key_cols):
    if df.empty:
        print(f"{label}: no data available")
        return
    available = [c for c in key_cols if c in df.columns]
    if not available:
        print(f"{label}: expected columns not present")
        return
    print(f"\n{label} ({len(df):,} rows, {df['report_id'].nunique() if 'report_id' in df.columns else '?'} reports)")
    print(df[available].describe(include='all').loc[['count','mean','std','min','max']].to_string())


acf_display_cols = [
    'report_id', 'acf_lag1', 'acf_lag_m', 'acf_max', 'ljung_box_pvalue', 'durbin_watson',
]

_show_acf_summary(bt_acf_summary,  'Backtest ACF summary (primary)', acf_display_cols)
_show_acf_summary(train_acf,        'Training ACF',                   acf_display_cols)
_show_acf_summary(prod_acf,         'Production ACF',                 acf_display_cols)

In [ ]:
# Show selected_m distribution across backtest ACF folds
if not bt_acf_fold.empty and 'candidate_m' in bt_acf_fold.columns:
    selected_m_counts = bt_acf_fold['candidate_m'].value_counts().sort_index()
    print("selected_m values seen across backtest ACF folds:")
    print(selected_m_counts.to_string())

    # Most common selected_m for labels
    selected_m = selected_m_counts.idxmax()
    print(f"\nMost common selected_m: {selected_m}")
else:
    selected_m = None
    print("Backtest ACF fold data unavailable — selected_m not determined.")

In [ ]:
# Ljung-Box significance rate
if not bt_acf_summary.empty and 'ljung_box_pvalue' in bt_acf_summary.columns:
    lb = bt_acf_summary['ljung_box_pvalue'].dropna()
    sig_rate = (lb < 0.05).mean()
    print(f"Backtest ACF summary — Ljung-Box significant (p<0.05): {sig_rate:.1%} of reports")
    print("Note: significance alone does not indicate an unusable model.")
else:
    print("Ljung-Box p-values not available in backtest ACF summary.")

## Bias Diagnostics

Bias measures systematic over- or under-forecasting. Sprint 5 computes bias
at three granularities:

- **Per fold**: mean residual per backtest fold.
- **Per horizon bucket**: short / medium / long horizon windows.
- **Summary**: cross-fold mean bias, median bias, and normalized bias.

**Normalized bias** = `sum(residuals) / sum(actuals)`. This is scale-free
and comparable across reports with different usage volumes. A value of +0.05
means the model underforecast by 5% of total actuals on average.

In [ ]:
def _show_bias_summary(df, label):
    if df.empty:
        print(f"{label}: no data available")
        return
    bias_cols = [c for c in ['mean_residual', 'median_residual', 'normalized_bias',
                              'bias_classification'] if c in df.columns]
    if not bias_cols:
        print(f"{label}: expected bias columns not present")
        return
    print(f"\n{label} ({len(df):,} rows)")
    numeric_cols = [c for c in bias_cols if c != 'bias_classification']
    if numeric_cols:
        print(df[numeric_cols].describe().loc[['count','mean','std','min','max']].to_string())
    if 'bias_classification' in df.columns:
        print("  bias_classification:", df['bias_classification'].value_counts().to_dict())


_show_bias_summary(bt_bias_summary, 'Backtest bias summary (primary)')
_show_bias_summary(train_bias,      'Training bias')
_show_bias_summary(prod_bias,       'Production bias')

In [ ]:
# Per-fold bias trend for the representative report
if not bt_bias_fold.empty and rep_healthy is not None and 'report_id' in bt_bias_fold.columns:
    rid = rep_healthy['report_id']
    fold_sub = bt_bias_fold[bt_bias_fold['report_id'] == rid].copy()
    if not fold_sub.empty and 'fold_number' in fold_sub.columns and 'mean_residual' in fold_sub.columns:
        fold_sub = fold_sub.sort_values('fold_number')
        fig, ax = plt.subplots(figsize=(8, 3))
        ax.bar(fold_sub['fold_number'], fold_sub['mean_residual'], color='steelblue', alpha=0.7)
        ax.axhline(0, color='black', lw=1)
        ax.set_title(f'Bias per fold — {rid} (healthy representative)')
        ax.set_xlabel('fold_number')
        ax.set_ylabel('mean residual (actual − forecast)')
        plt.tight_layout()
        plt.show()
    else:
        print(f"No per-fold bias data for report {rid}.")
else:
    print("No per-fold bias data or no healthy representative — skipping fold bias chart.")

## Residual Variance Stability

A model whose error variance grows or shrinks across folds may be encountering
regime changes. Sprint 5 tracks residual variance per fold and computes a
change ratio comparing recent vs. earlier folds.

In [ ]:
def _show_variance_stability(df, label):
    if df.empty:
        print(f"{label}: no data available")
        return
    var_cols = [c for c in ['residual_variance', 'variance_change_ratio',
                             'variance_status'] if c in df.columns]
    if not var_cols:
        print(f"{label}: variance stability columns not present")
        return
    print(f"\n{label} ({len(df):,} rows)")
    numeric_cols = [c for c in var_cols if c != 'variance_status']
    if numeric_cols:
        print(df[numeric_cols].describe().loc[['count','mean','std','min','max']].to_string())
    if 'variance_status' in df.columns:
        print("  variance_status:", df['variance_status'].value_counts().to_dict())


_show_variance_stability(bt_bias_summary, 'Backtest bias/variance summary')
_show_variance_stability(bt_bias_fold,    'Backtest bias/variance by fold')

## Outlier Diagnostics

Large individual misses can dominate reported error metrics. Sprint 5 identifies
outlier residuals using a **MAD (Median Absolute Deviation)** robust z-score:

```
robust_z = (residual - median) / (1.4826 * MAD)
```

Fallback order when MAD = 0: IQR → standard deviation → exact deviation.

Key metrics:
- `outlier_rate`: fraction of residuals flagged as outliers.
- `largest_miss`: absolute value of the single largest residual.
- `max_robust_z`: maximum robust z-score.

In [ ]:
def _show_outlier_summary(df, label):
    if df.empty:
        print(f"{label}: no data available")
        return
    out_cols = [c for c in ['outlier_rate', 'largest_miss', 'max_robust_z',
                             'outlier_classification'] if c in df.columns]
    if not out_cols:
        print(f"{label}: outlier columns not present")
        return
    print(f"\n{label} ({len(df):,} rows)")
    numeric_cols = [c for c in out_cols if c != 'outlier_classification']
    if numeric_cols:
        print(df[numeric_cols].describe().loc[['count','mean','std','min','max']].to_string())
    if 'outlier_classification' in df.columns:
        print("  outlier_classification:", df['outlier_classification'].value_counts().to_dict())


_show_outlier_summary(bt_outlier_summary, 'Backtest outlier summary (primary)')
_show_outlier_summary(train_outlier,       'Training outlier')
_show_outlier_summary(prod_outlier,        'Production outlier')

In [ ]:
# Outlier plot for a representative report
if not bt_errors.empty and 'residual' in bt_errors.columns and 'forecast_date' in bt_errors.columns:
    # Use healthy rep or first available report
    if rep_healthy is not None and 'report_id' in bt_errors.columns:
        rid = rep_healthy['report_id']
        sub = bt_errors[bt_errors['report_id'] == rid].copy()
    else:
        sub = bt_errors.copy()
        rid = 'all'

    sub['forecast_date'] = pd.to_datetime(sub['forecast_date'], errors='coerce')
    sub = sub.dropna(subset=['forecast_date', 'residual']).sort_values('forecast_date')

    if not sub.empty:
        # Compute a simple robust z for illustration
        median_r = sub['residual'].median()
        mad = (sub['residual'] - median_r).abs().median()
        scale = 1.4826 * mad if mad > 0 else (sub['residual'].std() or 1.0)
        sub['_rz'] = (sub['residual'] - median_r).abs() / scale
        outlier_mask = sub['_rz'] > 3.0

        fig, ax = plt.subplots(figsize=(10, 3))
        ax.scatter(sub['forecast_date'], sub['residual'],
                   s=8, color='steelblue', alpha=0.6, label='residual')
        ax.scatter(sub.loc[outlier_mask, 'forecast_date'],
                   sub.loc[outlier_mask, 'residual'],
                   s=40, color='red', zorder=5, label='outlier (|robust_z|>3)')
        ax.axhline(0, color='black', lw=1, ls='--')
        ax.set_title(f'Backtest residuals with outliers marked — report_id={rid}')
        ax.set_xlabel('forecast_date')
        ax.set_ylabel('residual (actual − forecast)')
        ax.legend(fontsize=8)
        plt.tight_layout()
        plt.show()
else:
    print("Backtest forecast errors not available — skipping outlier plot.")

## Distribution Diagnostics

Many SARIMA confidence intervals assume approximately normally distributed
residuals. Sprint 5 tests this with:

- **Skewness**: asymmetry of residuals.
- **Excess kurtosis**: tail heaviness relative to a normal distribution.
- **Jarque-Bera test**: joint test of skewness and kurtosis.
- **Shapiro-Wilk test** (when n < 5000): direct normality test.

**Important:** non-normality does NOT imply poor point forecasts. MASE and WAPE
are distribution-free metrics. Non-normality primarily affects the reliability of
prediction intervals, not point-forecast rankings.

In [ ]:
def _show_distribution_summary(df, label):
    if df.empty:
        print(f"{label}: no data available")
        return
    dist_cols = [c for c in ['skewness', 'excess_kurtosis', 'jarque_bera_pvalue',
                              'shapiro_pvalue', 'distribution_classification'] if c in df.columns]
    if not dist_cols:
        print(f"{label}: distribution columns not present")
        return
    print(f"\n{label} ({len(df):,} rows)")
    numeric_cols = [c for c in dist_cols if c != 'distribution_classification']
    if numeric_cols:
        print(df[numeric_cols].describe().loc[['count','mean','std','min','max']].to_string())
    if 'distribution_classification' in df.columns:
        print("  distribution_classification:", df['distribution_classification'].value_counts().to_dict())


_show_distribution_summary(bt_outlier_summary, 'Backtest distribution summary')
_show_distribution_summary(train_outlier,       'Training distribution')

In [ ]:
# Residual histogram for a representative report
if not bt_errors.empty and 'residual' in bt_errors.columns:
    if rep_healthy is not None and 'report_id' in bt_errors.columns:
        rid = rep_healthy['report_id']
        sub = bt_errors[bt_errors['report_id'] == rid]
    else:
        sub = bt_errors
        rid = 'all'

    vals = sub['residual'].dropna()
    if len(vals) > 2:
        skew_val = vals.skew()
        kurt_val = vals.kurt()
        fig, ax = plt.subplots(figsize=(7, 3))
        ax.hist(vals, bins=min(30, len(vals)//2 or 5), color='steelblue', alpha=0.7, edgecolor='white')
        ax.axvline(0, color='black', lw=1, ls='--')
        ax.set_title(
            f'Residual distribution — report_id={rid}\n'
            f'skewness={skew_val:.2f}, excess kurtosis={kurt_val:.2f}'
        )
        ax.set_xlabel('residual (actual − forecast)')
        ax.set_ylabel('count')
        plt.tight_layout()
        plt.show()
        print("Note: non-normality does not imply poor point forecasts.")
    else:
        print("Insufficient residuals for histogram.")
else:
    print("Backtest forecast errors not available — skipping histogram.")

## Prediction Interval Calibration

Prediction intervals aim for **nominal coverage** of 0.95 (alpha = 0.05).
Sprint 5 measures:

- **Coverage**: fraction of actuals falling within the 95% interval.
- **Coverage gap**: `nominal_coverage - observed_coverage` (positive = undercoverage).
- **Lower miss rate / upper miss rate**: direction of misses.
- **Interval width**: average width of the prediction interval.
- **Winkler interval score**: penalises both undercoverage and excessive width:

  ```
  Winkler = width + (2/alpha) * max(lower - actual, 0) + (2/alpha) * max(actual - upper, 0)
  ```

  Lower Winkler score is better.

In [ ]:
def _show_interval_summary(df, label):
    if df.empty:
        print(f"{label}: no data available")
        return
    int_cols = [c for c in ['coverage', 'coverage_gap', 'interval_width',
                              'winkler_score', 'lower_miss_rate', 'upper_miss_rate',
                              'interval_classification'] if c in df.columns]
    if not int_cols:
        print(f"{label}: interval columns not present")
        return
    print(f"\n{label} ({len(df):,} rows)")
    numeric_cols = [c for c in int_cols if c != 'interval_classification']
    if numeric_cols:
        print(df[numeric_cols].describe().loc[['count','mean','std','min','max']].to_string())
    if 'interval_classification' in df.columns:
        print("  interval_classification:", df['interval_classification'].value_counts().to_dict())


_show_interval_summary(bt_interval_summary, 'Backtest interval calibration summary (primary)')
_show_interval_summary(prod_interval,       'Production interval calibration')

In [ ]:
# Horizon-level coverage
if not bt_interval_fold.empty and 'horizon_bucket' in bt_interval_fold.columns and 'coverage' in bt_interval_fold.columns:
    horizon_cov = bt_interval_fold.groupby('horizon_bucket')['coverage'].mean().sort_index()
    print("Mean coverage by horizon bucket (backtest):")
    print(horizon_cov.to_string())
    print("Nominal target: 0.95")
else:
    print("Horizon-level coverage data not available.")

## Report-Level Model-Health Summary

The five diagnostic components are consolidated into one row per report in
`outputs/diagnostics/report_model_diagnostics_latest.csv`.

In [ ]:
if health_df.empty:
    print("report_model_diagnostics_latest.csv not available. Run the Sprint 5 pipeline first.")
else:
    print(f"Model-health rows  : {len(health_df):,}")
    print(f"Unique reports     : {health_df['report_id'].nunique() if 'report_id' in health_df.columns else '?'}")
    print()

    if 'model_diagnostic_status' in health_df.columns:
        print("Status distribution:")
        print(health_df['model_diagnostic_status'].value_counts().to_string())

    # Compact summary table: one row per report, key status columns only
    compact_cols = [c for c in [
        'report_id', 'model_diagnostic_status', 'evidence_status',
        'acf_status', 'bias_status', 'variance_status',
        'outlier_status', 'interval_status', 'recommended_action',
        'automatic_retraining_triggered',
    ] if c in health_df.columns]

    if compact_cols:
        print("\nCompact model-health table (first 10 rows):")
        print(health_df[compact_cols].head(10).to_string(index=False))

## Model-Health Classification

The classification follows a **deterministic precedence hierarchy** (highest priority first):

1. **`calculation_failed`** — diagnostic calculation failed despite valid source inputs.
2. **`insufficient_evidence`** — fewer than `MIN_VALID_BACKTEST_FOLDS=2` valid folds,
   or no production evidence and no backtest evidence.
3. **`poor`** — at least one critical or poor signal from any component; or severe
   production deterioration confirmed.
4. **`watch`** — one or more warning-level issues, or limited production evidence
   with acceptable backtest evidence.
5. **`healthy`** — no component in poor or warning status, sufficient evidence,
   no material production deterioration.

**Non-normality alone** does not make a model `poor`. A single significant
Ljung-Box, Jarque-Bera, or Shapiro-Wilk p-value does not override stronger
practical evidence.

**Recommended actions** are signals for human review only. `consider_retraining`
does not trigger any automatic pipeline action.

In [ ]:
if not health_df.empty and 'automatic_retraining_triggered' in health_df.columns:
    n_retrain = health_df['automatic_retraining_triggered'].sum()
    print(f"automatic_retraining_triggered = True: {n_retrain} reports")
    print("(Expected: 0 — no automatic retraining in Sprint 5)")
else:
    print("automatic_retraining_triggered column not available.")

## Representative Model-Health Case Studies

One representative report per health status is shown below.
Reports are selected deterministically (first by sort order of `report_id`).
If a category is unavailable, it is skipped gracefully.

In [ ]:
def _show_case_study(rep, label, health_df):
    if rep is None:
        print(f"{label}: No report currently meets this condition — skipping.")
        return
    rid = rep['report_id']
    print(f"\n{'='*60}")
    print(f"Case study — {label.upper()} — report_id={rid}")
    print('='*60)
    display_cols = [
        c for c in [
            'report_id', 'model_diagnostic_status', 'evidence_status',
            'acf_status', 'bias_status', 'outlier_status', 'interval_status',
            'recommended_action', 'automatic_retraining_triggered',
        ] if c in rep.index
    ]
    for col in display_cols:
        print(f"  {col:42s}: {rep[col]}")


_show_case_study(rep_healthy, 'healthy',               health_df)
_show_case_study(rep_watch,   'watch',                 health_df)
_show_case_study(rep_poor,    'poor',                  health_df)
_show_case_study(rep_insuf,   'insufficient_evidence', health_df)

## Relationship to Model Selection

Sprint 5 diagnostics are **downstream of** and **independent from** model selection:

- Model selection (`src/models/selection.py`) ranks candidates by **MASE** across
  backtest folds. MASE is a scale-free, distribution-free accuracy metric.
- Diagnostics evaluate the **already-selected** model for operational health.
- A `poor` or `watch` model-health status does NOT change the selected model.
- MASE rankings are not re-computed or re-ordered by any Sprint 5 module.

The rationale: MASE already captures the key accuracy signal. Diagnostics
provide additional context for human review — they are a layer of explanation,
not a second selection stage.

## Relationship to Later Sprints

`outputs/diagnostics/report_model_diagnostics_latest.csv` is a **primary input**
for downstream consumers:

| Consumer | How model-health is used |
|----------|-------------------------|
| Notebook 07 (user analytics) | Joins model health to user-level usage patterns |
| Notebook 08 (GenAI insights) | Uses `model_diagnostic_status` and `recommended_action` in prompt context |
| Streamlit demo app | Displays model-health badge per report; filters by status |
| Future monitoring pipeline | May extend Sprint 5 production fields as observation counts grow |

The `diagnostic_evidence_status` field indicates whether there was sufficient
evidence for each component. Consumers should check this field before relying
on per-component status values.

## Limitations

1. **Short series:** many reports have fewer than 90 days of history. Diagnostic
   statistics computed on < 30 residuals are noisy and should be interpreted
   cautiously. The `evidence_status` field flags these cases.

2. **Production sparsity:** `production_forecast_errors_latest.csv` may contain
   very few rows early in the pipeline lifecycle. The `MIN_PRODUCTION_ERROR_COUNT=10`
   threshold prevents premature conclusions.

3. **Statistical tests:** Ljung-Box, Jarque-Bera, and Shapiro-Wilk p-values are
   sensitive to sample size. On large datasets, trivially small effects become
   significant. On tiny datasets, large effects may not reach significance.

4. **Interval calibration validity:** Winkler scores and coverage rates are only
   meaningful when prediction intervals were saved. Not all models produce intervals.

5. **No causal diagnosis:** diagnostics identify that a problem exists; they do
   not automatically identify the cause (structural break, data-quality issue,
   wrong seasonal period, etc.).

6. **No automatic retraining:** `automatic_retraining_triggered` is always `False`
   in Sprint 5. Human review of `consider_retraining` flags is required.

7. **Synthetic data:** the current portfolio uses synthetic report usage data.
   Diagnostic thresholds were calibrated for a reasonable synthetic distribution
   and may need tuning on real-world data.